# Pyo fundamentals

A "hello world" tour of [pyo](http://ajaxsoundstudio.com/pyodoc/), building up from a single sine wave to a small synth. Each section adds one new idea on top of the last.

Run cells top to bottom. Sound keeps playing after a cell runs until you `.stop()` it or run something else `.out()` on the same variable.

## 1. Boot the server

Everything in pyo needs an audio server running first — think of it as turning on the sound card.

In [1]:
from pyo.lib.server import Server

# duplex=0: output only, no audio input. Avoids boot issues when your input
# device (e.g. a Bluetooth headset) has fewer channels than pyo requests by
# default, which can otherwise leave the server silent on first boot.
s = Server(duplex=0).boot()
s.start()

Pyo warning: Portmidi warning: no midi device found!
Portmidi closed.


## 2. Hello, sine wave

The simplest possible sound: one oscillator, one frequency, `.out()` to send it to your speakers.

In [2]:
from pyo.lib.generators import Sine

hello = Sine(freq=440, mul=1.2).out()

In [3]:
hello.stop()

< Instance of Sine class >

## 3. Changing parameters live

Objects are just Python objects — set attributes on them while they're playing to hear the change.

In [4]:
hello = Sine(freq=440, mul=0.3).out()

In [5]:
hello.freq = 220  # drop an octave

In [6]:
hello.freq = 880  # up two octaves from there

In [7]:
hello.stop()

< Instance of Sine class >

## 4. More than one voice

Pass a list instead of a single number and pyo plays multiple voices at once (one per channel/value). This is how you get a chord out of a single `Sine` object.

In [8]:
chord = Sine(freq=[220, 277.18, 329.63], mul=0.2).out()  # A minor-ish triad

In [9]:
chord.stop()

< Instance of Sine class >

## 5. A wave table instead of a pure sine

`Osc` reads its waveform out of a **table** rather than computing a fixed shape, which is what lets you swap timbres. Compare a square-ish table to a smooth harmonic one.

In [10]:
from pyo.lib.tables import SquareTable, HarmTable
from pyo.lib.tableprocess import Osc

# wav = SquareTable()
wav = HarmTable([1, 0, 0.33, 0, 0.2, 0, 0.143, 0, 0.111])

osc = Osc(table=wav, freq=30, mul=0.3).out()

In [11]:
osc.stop()

< Instance of Osc class >

## 6. A steady beat

`Metro` fires a trigger at a regular interval instead of playing audio itself — it's a clock. On its own it's silent; you need something listening to it.

In [12]:
from pyo.lib.triggers import Metro

clock = Metro(time=0.25).play()  # fires every quarter second

## 7. Turning triggers into notes

Feed the `Metro`'s triggers into `TrigEnv` (a one-shot envelope) so each tick becomes a short amplitude "pluck" instead of a held tone.

In [13]:
from pyo.lib.tables import CosTable
from pyo.lib.triggers import TrigEnv

envelope_shape = CosTable([(0, 0), (100, 1), (8191, 0)])  # quick attack, long decay
pluck_amp = TrigEnv(clock, table=envelope_shape, dur=0.2, mul=0.4)

beep = Osc(table=SquareTable(), freq=330, mul=pluck_amp).out()

In [14]:
beep.stop()
clock.stop()

< Instance of Metro class >

## 8. Randomised notes on the beat

`TrigXnoiseMidi` picks a new (quantised, MIDI-style) pitch every time it receives a trigger — combine it with the `Metro` clock for a simple random note generator.

In [15]:
from pyo.lib.triggers import TrigXnoiseMidi

clock = Metro(time=0.2).play()
pluck_amp = TrigEnv(clock, table=envelope_shape, dur=0.2, mul=0.4)
pitch = TrigXnoiseMidi(clock, dist=0, scale=1, mrange=(48, 72))  # scale=1 -> Hz output

random_notes = Osc(table=SquareTable(), freq=pitch, mul=pluck_amp).out()

In [16]:
random_notes.stop()
clock.stop()

< Instance of Metro class >

## 9. A wobble with LFO, and a proper synth voice with FM

`LFO` is a slow oscillator used to modulate another parameter rather than to make sound directly. Here it drifts an `FM` synth's modulation index for a more alive, evolving tone.

In [17]:
from pyo.lib.generators import LFO, FM

wobble = LFO(freq=0.3, sharp=0.5, type=0, mul=3, add=5)  # slowly drifting 2..8

synth = FM(carrier=110, ratio=0.5, index=wobble, mul=0.3).out()

In [18]:
synth.stop()

< Instance of FM class >

## 10. Putting it together: a triggered FM synth line

Same building blocks as above — a clock, an envelope, a random pitch source — but driving an `FM` synth instead of a plain `Osc`. This is basically where `02_Pyo_demo.ipynb` picks up.

In [19]:
clock = Metro(time=0.15, poly=4).play()
amp_shape = CosTable([(0, 0), (100, 1), (500, 0.3), (8191, 0)])
amp = TrigEnv(clock, table=amp_shape, dur=0.3, mul=0.4)
pitch = TrigXnoiseMidi(clock, dist=0, scale=1, mrange=(48, 84))

line = FM(carrier=pitch, ratio=0.25, index=amp, mul=0.4).out()

In [20]:
line.stop()
clock.stop()

< Instance of Metro class >

## 11. Interactive control

A slider driving a live parameter, same trick used at the end of `02_Pyo_demo.ipynb`.

In [ ]:
from ipywidgets import IntSlider, interact

hello = Sine(freq=440, mul=0.3).out()


def set_freq(value):
    hello.freq = value


interact(set_freq, value=IntSlider(min=110, max=880, step=10, value=440))

In [ ]:
hello.stop()

## 12. Shut it down

In [ ]:
s.stop()